**Приложение D – Автодифференцирование (Autodiff)**

_Этот ноутбук содержит игрушечные реализации различных методов автодифференцирования, объясняющие, как они работают._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml2/blob/master/extra_autodiff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml2/blob/master/extra_autodiff.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Настройка

# Введение

Допустим, нужно вычислить градиенты функции $f(x,y)=x^2y + y + 2$ по параметрам x и y:

In [1]:
def f(x,y):
    return x*x*y + y + 2

Один подход — решить задачу аналитически:

$\dfrac{\partial f}{\partial x} = 2xy$

$\dfrac{\partial f}{\partial y} = x^2 + 1$

In [2]:
def df(x,y):
    return 2*x*y, x*x + 1

Например, $\dfrac{\partial f}{\partial x}(3,4) = 24$ и $\dfrac{\partial f}{\partial y}(3,4) = 10$.

In [3]:
df(3, 4)

(24, 10)

Отлично! Можно также найти уравнения для производных второго порядка (их также называют гессианами, Hessians):

$\dfrac{\partial^2 f}{\partial x \partial x} = \dfrac{\partial (2xy)}{\partial x} = 2y$

$\dfrac{\partial^2 f}{\partial x \partial y} = \dfrac{\partial (2xy)}{\partial y} = 2x$

$\dfrac{\partial^2 f}{\partial y \partial x} = \dfrac{\partial (x^2 + 1)}{\partial x} = 2x$

$\dfrac{\partial^2 f}{\partial y \partial y} = \dfrac{\partial (x^2 + 1)}{\partial y} = 0$

При x=3 и y=4 эти гессианы равны соответственно 8, 6, 6, 0. Вычислим их по приведённым выше уравнениям:

In [4]:
def d2f(x, y):
    return [2*y, 2*x], [2*x, 0]

In [5]:
d2f(3, 4)

([8, 6], [6, 0])

Отлично, но для этого нужна некоторая математическая работа. В данном случае это не так уж сложно, но для глубокой нейронной сети вычислять производные таким образом практически невозможно. Так что рассмотрим разные способы это автоматизировать!

# Численное дифференцирование

Здесь мы вычисляем приближение градиентов по формуле $\dfrac{\partial f}{\partial x} = \displaystyle{\lim_{\epsilon \to 0}}\dfrac{f(x+\epsilon, y) - f(x, y)}{\epsilon}$ (и аналогичное определение есть для $\dfrac{\partial f}{\partial y}$).

In [6]:
def gradients(func, vars_list, eps=0.0001):
    partial_derivatives = []
    base_func_eval = func(*vars_list)
    for idx in range(len(vars_list)):
        tweaked_vars = vars_list[:]
        tweaked_vars[idx] += eps
        tweaked_func_eval = func(*tweaked_vars)
        derivative = (tweaked_func_eval - base_func_eval) / eps
        partial_derivatives.append(derivative)
    return partial_derivatives

In [7]:
def df(x, y):
    return gradients(f, [x, y])

In [8]:
df(3, 4)

[24.000400000048216, 10.000000000047748]

Работает хорошо!

Хорошая новость в том, что гессианы вычислить довольно легко. Сначала создадим функции, вычисляющие частные производные первого порядка (их также называют якобианами, Jacobians):

In [9]:
def dfdx(x, y):
    return gradients(f, [x,y])[0]

def dfdy(x, y):
    return gradients(f, [x,y])[1]

dfdx(3., 4.), dfdy(3., 4.)

(24.000400000048216, 10.000000000047748)

Теперь можно просто применить к этим функциям функцию `gradients()`:

In [10]:
def d2f(x, y):
    return [gradients(dfdx, [x, y]), gradients(dfdy, [x, y])]

In [11]:
d2f(3, 4)

[[7.999999951380232, 6.000099261882497],
 [6.000099261882497, -1.4210854715202004e-06]]

Итак, всё работает хорошо, но результат приближённый, а для вычисления градиентов функции по $n$ переменным нужно вызвать эту функцию $n$ раз. В глубоких нейронных сетях часто тысячи параметров, которые нужно подстраивать градиентным спуском (для этого нужно вычислять градиенты функции потерь по каждому из параметров), поэтому такой подход был бы слишком медленным.

## Реализация игрушечного вычислительного графа

Вместо этого численного подхода реализуем несколько методов символьного автодифференцирования. Для этого понадобится определить классы, представляющие константы, переменные и операции.

In [12]:
class Const(object):
    def __init__(self, value):
        self.value = value
    def evaluate(self):
        return self.value
    def __str__(self):
        return str(self.value)

class Var(object):
    def __init__(self, name, init_value=0):
        self.value = init_value
        self.name = name
    def evaluate(self):
        return self.value
    def __str__(self):
        return self.name

class BinaryOperator(object):
    def __init__(self, a, b):
        self.a = a
        self.b = b

class Add(BinaryOperator):
    def evaluate(self):
        return self.a.evaluate() + self.b.evaluate()
    def __str__(self):
        return "{} + {}".format(self.a, self.b)

class Mul(BinaryOperator):
    def evaluate(self):
        return self.a.evaluate() * self.b.evaluate()
    def __str__(self):
        return "({}) * ({})".format(self.a, self.b)

Хорошо, теперь можно построить вычислительный граф, представляющий функцию $f$:

In [13]:
x = Var("x")
y = Var("y")
f = Add(Mul(Mul(x, x), y), Add(y, Const(2))) # f(x,y) = x²y + y + 2

И можно выполнить этот граф, чтобы вычислить $f$ в любой точке, например $f(3, 4)$.

In [14]:
x.value = 3
y.value = 4
f.evaluate()

42

Отлично, он нашёл главный ответ (на главный вопрос жизни, Вселенной и всего такого).

## Вычисление градиентов

Все методы автодифференцирования, описанные ниже, основаны на *цепном правиле* (chain rule).

Допустим, у нас есть две функции $u$ и $v$, мы последовательно применяем их к некоторому входу $x$ и получаем результат $z$. Тогда $z = v(u(x))$, что можно переписать как $z = v(s)$ и $s = u(x)$. Теперь можно применить цепное правило, чтобы получить частную производную выхода $z$ по входу $x$:

$ \dfrac{\partial z}{\partial x} = \dfrac{\partial s}{\partial x} \cdot \dfrac{\partial z}{\partial s}$

Если же $z$ — выход последовательности функций с промежуточными выходами $s_1, s_2, ..., s_n$, цепное правило по-прежнему применимо:

$ \dfrac{\partial z}{\partial x} = \dfrac{\partial s_1}{\partial x} \cdot \dfrac{\partial s_2}{\partial s_1} \cdot \dfrac{\partial s_3}{\partial s_2} \cdot \dots \cdot \dfrac{\partial s_{n-1}}{\partial s_{n-2}} \cdot \dfrac{\partial s_n}{\partial s_{n-1}} \cdot \dfrac{\partial z}{\partial s_n}$

При автодифференцировании в прямом режиме (forward mode) алгоритм вычисляет эти члены «вперёд» (то есть в том же порядке, что и вычисления, нужные для получения выхода $z$), то есть слева направо: сначала $\dfrac{\partial s_1}{\partial x}$, затем $\dfrac{\partial s_2}{\partial s_1}$ и так далее. При автодифференцировании в обратном режиме (reverse mode) алгоритм вычисляет эти члены «назад», справа налево: сначала $\dfrac{\partial z}{\partial s_n}$, затем $\dfrac{\partial s_n}{\partial s_{n-1}}$ и так далее.

Например, допустим, нужно вычислить производную функции $z(x)=\sin(x^2)$ при x=3 с помощью автодифференцирования в прямом режиме. Алгоритм сначала вычислит частную производную $\dfrac{\partial s_1}{\partial x}=\dfrac{\partial x^2}{\partial x}=2x=6$. Затем он вычислит $\dfrac{\partial z}{\partial x}=\dfrac{\partial s_1}{\partial x}\cdot\dfrac{\partial z}{\partial s_1}= 6 \cdot \dfrac{\partial \sin(s_1)}{\partial s_1}=6 \cdot \cos(s_1)=6 \cdot \cos(3^2)\approx-5.46$.

Проверим этот результат с помощью определённой ранее функции `gradients()`:

In [15]:
from math import sin

def z(x):
    return sin(x**2)

gradients(z, [3])

[-5.46761419430053]

Выглядит хорошо. Теперь сделаем то же самое с помощью автодифференцирования в обратном режиме. На этот раз алгоритм начнёт справа и вычислит $\dfrac{\partial z}{\partial s_1} = \dfrac{\partial \sin(s_1)}{\partial s_1}=\cos(s_1)=\cos(3^2)\approx -0.91$. Затем он вычислит $\dfrac{\partial z}{\partial x}=\dfrac{\partial s_1}{\partial x}\cdot\dfrac{\partial z}{\partial s_1} \approx \dfrac{\partial s_1}{\partial x} \cdot -0.91 = \dfrac{\partial x^2}{\partial x} \cdot -0.91=2x \cdot -0.91 = 6\cdot-0.91=-5.46$.

Конечно, оба подхода дают один и тот же результат (с точностью до ошибок округления), а при одном входе и одном выходе требуют одинакового объёма вычислений. Но когда входов или выходов несколько, их производительность может сильно различаться. Действительно, если входов много, самые правые члены понадобятся для вычисления частных производных по каждому входу, поэтому разумно вычислить эти правые члены первыми. Это означает использование обратного режима. Так самые правые члены можно вычислить всего один раз и использовать для вычисления всех частных производных. И наоборот, если выходов много, обычно предпочтительнее прямой режим, поскольку самые левые члены можно вычислить один раз для частных производных разных выходов. В глубоком обучении параметров модели обычно тысячи, то есть входов много, а выходов мало. На самом деле во время обучения выход обычно всего один — функция потерь. Именно поэтому в TensorFlow и всех крупных библиотеках глубокого обучения используется автодифференцирование в обратном режиме.

У обратного режима есть ещё одна сложность: при вычислении $\dfrac{\partial s_{i+1}}{\partial s_i}$ обычно требуется значение $s_i$, а для вычисления $s_i$ нужно сначала вычислить $s_{i-1}$, для чего нужно вычислить $s_{i-2}$, и так далее. Поэтому, по сути, сначала нужен прямой проход по сети, чтобы вычислить $s_1$, $s_2$, $s_3$, $\dots$, $s_{n-1}$ и $s_n$, и только затем алгоритм может вычислять частные производные справа налево. Хранение всех промежуточных значений $s_i$ в оперативной памяти иногда становится проблемой, особенно при работе с изображениями и при использовании GPU, у которых память часто ограничена: чтобы смягчить проблему, можно уменьшить число слоёв нейронной сети или настроить TensorFlow так, чтобы он выгружал эти значения из памяти GPU в оперативную память CPU. Другой подход — кешировать только каждое второе промежуточное значение: $s_1$, $s_3$, $s_5$, $\dots$, $s_{n-4}$, $s_{n-2}$ и $s_n$. Это означает, что при вычислении частных производных, если промежуточного значения $s_i$ нет, алгоритму придётся вычислить его заново по предыдущему промежуточному значению $s_{i-1}$. Так процессорное время обменивается на память (если интересно, посмотрите [эту статью](https://pdfs.semanticscholar.org/f61e/9fd5a4878e1493f7a6b03774a61c17b7e9a4.pdf)).

### Автодифференцирование в прямом режиме

In [16]:
Const.gradient = lambda self, var: Const(0)
Var.gradient = lambda self, var: Const(1) if self is var else Const(0)
Add.gradient = lambda self, var: Add(self.a.gradient(var), self.b.gradient(var))
Mul.gradient = lambda self, var: Add(Mul(self.a, self.b.gradient(var)), Mul(self.a.gradient(var), self.b))

x = Var(name="x", init_value=3.)
y = Var(name="y", init_value=4.)
f = Add(Mul(Mul(x, x), y), Add(y, Const(2))) # f(x,y) = x²y + y + 2

dfdx = f.gradient(x)  # 2xy
dfdy = f.gradient(y)  # x² + 1

In [17]:
dfdx.evaluate(), dfdy.evaluate()

(24.0, 10.0)

Поскольку выход метода `gradient()` полностью символьный, мы не ограничены производными первого порядка: можно вычислять и производные второго порядка, и так далее:

In [18]:
d2fdxdx = dfdx.gradient(x) # 2y
d2fdxdy = dfdx.gradient(y) # 2x
d2fdydx = dfdy.gradient(x) # 2x
d2fdydy = dfdy.gradient(y) # 0

In [19]:
[[d2fdxdx.evaluate(), d2fdxdy.evaluate()],
 [d2fdydx.evaluate(), d2fdydy.evaluate()]]

[[8.0, 6.0], [6.0, 0.0]]

Обратите внимание, что теперь результат точный, а не приближённый (конечно, в пределах точности чисел с плавающей точкой на машине).

### Автодифференцирование в прямом режиме с помощью дуальных чисел

Изящный способ применить автодифференцирование в прямом режиме — использовать [дуальные числа](https://en.wikipedia.org/wiki/Dual_number). Вкратце, дуальное число $z$ имеет вид $z = a + b\epsilon$, где $a$ и $b$ — вещественные числа, а $\epsilon$ — бесконечно малое число, положительное, но меньшее всех вещественных чисел, такое, что $\epsilon^2=0$.
Можно показать, что $f(x + \epsilon) = f(x) + \dfrac{\partial f}{\partial x}\epsilon$, поэтому, просто вычислив $f(x + \epsilon)$, мы получаем и значение $f(x)$, и частную производную $f$ по $x$.

У дуальных чисел свои правила арифметики, обычно вполне естественные. Например:

**Сложение**

$(a_1 + b_1\epsilon) + (a_2 + b_2\epsilon) = (a_1 + a_2) + (b_1 + b_2)\epsilon$

**Вычитание**

$(a_1 + b_1\epsilon) - (a_2 + b_2\epsilon) = (a_1 - a_2) + (b_1 - b_2)\epsilon$

**Умножение**

$(a_1 + b_1\epsilon) \times (a_2 + b_2\epsilon) = (a_1 a_2) + (a_1 b_2 + a_2 b_1)\epsilon + b_1 b_2\epsilon^2 = (a_1 a_2) + (a_1b_2 + a_2b_1)\epsilon$

**Деление**

$\dfrac{a_1 + b_1\epsilon}{a_2 + b_2\epsilon} = \dfrac{a_1 + b_1\epsilon}{a_2 + b_2\epsilon} \cdot \dfrac{a_2 - b_2\epsilon}{a_2 - b_2\epsilon} = \dfrac{a_1 a_2 + (b_1 a_2 - a_1 b_2)\epsilon - b_1 b_2\epsilon^2}{{a_2}^2 + (a_2 b_2 - a_2 b_2)\epsilon - {b_2}^2\epsilon} = \dfrac{a_1}{a_2} + \dfrac{a_1 b_2 - b_1 a_2}{{a_2}^2}\epsilon$

**Возведение в степень**

$(a + b\epsilon)^n = a^n + (n a^{n-1}b)\epsilon$

и т. д.

Создадим класс, представляющий дуальные числа, и реализуем несколько операций (сложение и умножение). При желании можете попробовать добавить другие.

In [20]:
class DualNumber(object):
    def __init__(self, value=0.0, eps=0.0):
        self.value = value
        self.eps = eps
    def __add__(self, b):
        return DualNumber(self.value + self.to_dual(b).value,
                          self.eps + self.to_dual(b).eps)
    def __radd__(self, a):
        return self.to_dual(a).__add__(self)
    def __mul__(self, b):
        return DualNumber(self.value * self.to_dual(b).value,
                          self.eps * self.to_dual(b).value + self.value * self.to_dual(b).eps)
    def __rmul__(self, a):
        return self.to_dual(a).__mul__(self)
    def __str__(self):
        if self.eps:
            return "{:.1f} + {:.1f}ε".format(self.value, self.eps)
        else:
            return "{:.1f}".format(self.value)
    def __repr__(self):
        return str(self)
    @classmethod
    def to_dual(cls, n):
        if hasattr(n, "value"):
            return n
        else:
            return cls(n)

$3 + (3 + 4 \epsilon) = 6 + 4\epsilon$

In [21]:
3 + DualNumber(3, 4)

6.0 + 4.0ε

$(3 + 4ε)\times(5 + 7ε)$ = $3 \times 5 + 3 \times 7ε + 4ε \times 5 + 4ε \times 7ε$ = $15 + 21ε + 20ε + 28ε^2$ = $15 + 41ε + 28 \times 0$ = $15 + 41ε$

In [22]:
DualNumber(3, 4) * DualNumber(5, 7)

15.0 + 41.0ε

Теперь посмотрим, работают ли дуальные числа с нашим игрушечным вычислительным фреймворком:

In [23]:
x.value = DualNumber(3.0)
y.value = DualNumber(4.0)

f.evaluate()

42.0

Да, работает. Теперь с их помощью вычислим частные производные $f$ по $x$ и $y$ при x=3 и y=4:

In [24]:
x.value = DualNumber(3.0, 1.0)  # 3 + ε
y.value = DualNumber(4.0)       # 4

dfdx = f.evaluate().eps

x.value = DualNumber(3.0)       # 3
y.value = DualNumber(4.0, 1.0)  # 4 + ε

dfdy = f.evaluate().eps

In [25]:
dfdx

24.0

In [26]:
dfdy

10.0

Отлично! Однако в этой реализации мы ограничены производными первого порядка.
Теперь рассмотрим обратный режим.

### Автодифференцирование в обратном режиме

Перепишем наш игрушечный фреймворк, добавив автодифференцирование в обратном режиме:

In [27]:
class Const(object):
    def __init__(self, value):
        self.value = value
    def evaluate(self):
        return self.value
    def backpropagate(self, gradient):
        pass
    def __str__(self):
        return str(self.value)

class Var(object):
    def __init__(self, name, init_value=0):
        self.value = init_value
        self.name = name
        self.gradient = 0
    def evaluate(self):
        return self.value
    def backpropagate(self, gradient):
        self.gradient += gradient
    def __str__(self):
        return self.name

class BinaryOperator(object):
    def __init__(self, a, b):
        self.a = a
        self.b = b

class Add(BinaryOperator):
    def evaluate(self):
        self.value = self.a.evaluate() + self.b.evaluate()
        return self.value
    def backpropagate(self, gradient):
        self.a.backpropagate(gradient)
        self.b.backpropagate(gradient)
    def __str__(self):
        return "{} + {}".format(self.a, self.b)

class Mul(BinaryOperator):
    def evaluate(self):
        self.value = self.a.evaluate() * self.b.evaluate()
        return self.value
    def backpropagate(self, gradient):
        self.a.backpropagate(gradient * self.b.value)
        self.b.backpropagate(gradient * self.a.value)
    def __str__(self):
        return "({}) * ({})".format(self.a, self.b)

In [28]:
x = Var("x", init_value=3)
y = Var("y", init_value=4)
f = Add(Mul(Mul(x, x), y), Add(y, Const(2))) # f(x,y) = x²y + y + 2

result = f.evaluate()
f.backpropagate(1.0)

In [29]:
print(f)

((x) * (x)) * (y) + y + 2


In [30]:
result

42

In [31]:
x.gradient

24.0

In [32]:
y.gradient

10.0

И снова в этой реализации выходы — просто числа, а не символьные выражения, поэтому мы ограничены производными первого порядка. Однако можно было бы сделать так, чтобы методы `backpropagate()` возвращали символьные выражения, а не значения (например, `Add(2,3)` вместо 5). Тогда можно было бы вычислять градиенты второго порядка (и выше). Именно так поступает TensorFlow, как и все крупные библиотеки, реализующие автодифференцирование.

### Автодифференцирование в обратном режиме с помощью TensorFlow

In [33]:
import tensorflow as tf

In [34]:
x = tf.Variable(3.)
y = tf.Variable(4.)

with tf.GradientTape() as tape:
    f = x*x*y + y + 2

jacobians = tape.gradient(f, [x, y])
jacobians

[<tf.Tensor: shape=(), dtype=float32, numpy=24.0>,
 <tf.Tensor: shape=(), dtype=float32, numpy=10.0>]

Поскольку всё символьное, можно вычислять производные второго порядка и выше:

In [35]:
x = tf.Variable(3.)
y = tf.Variable(4.)

with tf.GradientTape(persistent=True) as tape:
    f = x*x*y + y + 2
    df_dx, df_dy = tape.gradient(f, [x, y])

d2f_d2x, d2f_dydx = tape.gradient(df_dx, [x, y])
d2f_dxdy, d2f_d2y = tape.gradient(df_dy, [x, y])
del tape

hessians = [[d2f_d2x, d2f_dydx], [d2f_dxdy, d2f_d2y]]
hessians

[[<tf.Tensor: shape=(), dtype=float32, numpy=8.0>,
  <tf.Tensor: shape=(), dtype=float32, numpy=6.0>],
 [<tf.Tensor: shape=(), dtype=float32, numpy=6.0>, None]]

Обратите внимание: если вычислять производную тензора по переменной, от которой он не зависит, функция `gradient()` вернёт не 0.0, а `None`.

Вот и всё, друзья! Надеюсь, вам понравился этот ноутбук.